# CIS 6211 – Foundations of Data Science
## Lab 3: Data Cleaning & Preprocessing

**Student Name:** Rana Sultan Alhinidy  
**Course:** CIS 6211 | King Khalid University


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")

### 📌 Cell Explanation
This cell imports the required libraries:
- **pandas** is used for loading and manipulating the dataset.
- **numpy** provides numerical operations such as the `where()` function used for outlier capping.
- **matplotlib** and **seaborn** are used for data visualization (box plots, histograms).
- `sns.set(style='whitegrid')` applies a clean visual style to all seaborn plots.

## Step 1 – Load Adult Dataset

In [ ]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country",
    "income"
]

df = pd.read_csv(
    "adult.data",
    names=columns,
    na_values="?",
    skipinitialspace=True
)

df.head()

### 📌 Cell Explanation
The **UCI Adult Income** dataset is loaded from a CSV file. This dataset contains demographic and occupational information used to predict whether a person earns above or below $50K annually.

Key loading parameters:
- `names=columns` — the file has no header row, so column names are specified manually.
- `na_values='?'` — question marks in the original file represent missing values; this converts them to `NaN` so pandas can handle them properly.
- `skipinitialspace=True` — removes leading whitespace from string values, which is common in this dataset.

## Step 2 – Identify Missing Values

In [ ]:
df.isnull().sum()

### 📌 Cell Explanation
The number of missing values in each column is counted. `isnull()` returns a True/False DataFrame, and `sum()` counts the `True` values per column.

The columns with missing values are:
- **workclass** — 1,836 missing
- **occupation** — 1,843 missing
- **native_country** — 583 missing

This step is essential to assess the extent of the missing data problem and decide on the appropriate handling strategy.

## Step 3 – Handle Missing Values

In [ ]:
# Drop rows with missing values (small proportion)
df.dropna(inplace=True)

### 📌 Cell Explanation
All rows containing missing values are removed using `dropna()`.

The decision to **drop rather than impute** is justified here because:
- The dataset has over 30,000 rows, and the missing values represent a small proportion (~7%).
- Dropping these rows preserves data integrity without introducing artificial values.

`inplace=True` modifies the original DataFrame directly without creating a new copy, saving memory.

## Step 4 – Check Data Types

In [ ]:
df.dtypes

### 📌 Cell Explanation
The data type of each column is displayed to verify correctness:
- Numeric columns such as `age` and `hours_per_week` should be `int64`.
- Categorical columns such as `workclass` and `education` should be `object` (string).

Verifying data types is important because incorrect types cause errors in calculations. For example, if `age` were stored as a string, computing its mean would fail.

## Step 5 – Detect Outliers

In [ ]:
sns.boxplot(x=df["capital_gain"])
plt.title("Capital Gain (Highly Skewed)")
plt.show()

### 📌 Cell Explanation
A box plot is drawn for the `capital_gain` column to detect outliers.

The box plot shows:
- **Q1** (25th percentile), **Median** (Q2), **Q3** (75th percentile) as the box.
- Points beyond **1.5 × IQR** are plotted as individual dots — these are outliers.

The `capital_gain` column is **heavily right-skewed**: most values are zero, but a few records have extremely high values (up to 99,999), which would distort any statistical analysis.

## Step 6 – Handle Outliers (Capping / Winsorization)

In [ ]:
cap = df["capital_gain"].quantile(0.99)
df["capital_gain"] = np.where(df["capital_gain"] > cap, cap, df["capital_gain"])

### 📌 Cell Explanation
Outliers are handled using the **Winsorization (capping)** method:

1. The **99th percentile** is computed — this is the value below which 99% of the data falls.
2. Any value exceeding this threshold is replaced with the threshold itself using `np.where()`.

**Why capping instead of deletion?**  
Capping preserves all rows while reducing the impact of extreme values on statistical analysis. Deleting outliers would reduce the dataset size unnecessarily, especially when the outlier might represent a real (not erroneous) value.

## Step 7 – Final Dataset Shape

In [ ]:
df.shape

### 📌 Cell Explanation
The final dimensions of the cleaned DataFrame are displayed (rows × columns).

The output `(30162, 15)` means:
- **30,162 rows** remain after dropping missing values (original: ~32,561 rows).
- **15 columns** remain unchanged.

Documenting the final shape is important for tracking how much data was removed at each cleaning step and confirming the dataset is ready for modeling.